In [1]:
# Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Load data
data = pd.read_csv("../data/cleaned_data.csv")

# 2. Select features (X) and target (y)
X = data[['Distance_km', 'Preparation_Time_min', 'Traffic_Level', 'Weather']]
y = data['Delivery_Time_min']

# 3. Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Define columns
numerical_cols = ['Distance_km', 'Preparation_Time_min']
categorical_cols = ['Traffic_Level', 'Weather']

# 5. Preprocessing
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(drop='first'), categorical_cols)
])

# 6. Feature selection
feature_selector = SelectKBest(score_func=f_regression, k='all')

# 7. Define models
models = {
    'RandomForest': RandomForestRegressor(random_state=42),
    'SVR': SVR()
}

# 8. Define hyperparameter grids
param_grids = {
    'RandomForest': {
        'model__n_estimators': [50, 100],
        'model__max_depth': [5, 10, None],
        'model__min_samples_split': [2, 5, 10]
    },
    'SVR': {
        'model__kernel': ['linear', 'rbf'],
        'model__C': [1, 10]
    }
}

# 9. Loop for both models
for name, model in models.items():
    print(f"\n=== Training {name} ===")
    
    # Create pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('feature_selection', feature_selector),
        ('model', model)
    ])
    
    # Grid search
    grid = GridSearchCV(pipeline, param_grids[name], cv=5, scoring='r2')
    grid.fit(X_train, y_train)
    
    # Predictions
    y_pred = grid.predict(X_test)
    
    # Evaluate
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Print results
    print(f"mae: {mae}")
    print(f"r2: {r2}")
    print("meilleur hyperparametre:", grid.best_params_)
    print("meilleur score R²:", grid.best_score_)
    print("R² sur le test set:", r2)



=== Training RandomForest ===
mae: 7.274617919335919
r2: 0.7563214489055523
meilleur hyperparametre: {'model__max_depth': 10, 'model__min_samples_split': 10, 'model__n_estimators': 100}
meilleur score R²: 0.689908851816764
R² sur le test set: 0.7563214489055523

=== Training SVR ===
mae: 5.912448254281191
r2: 0.8172989134693437
meilleur hyperparametre: {'model__C': 10, 'model__kernel': 'linear'}
meilleur score R²: 0.7459691651000651
R² sur le test set: 0.8172989134693437
